In [1]:
from __future__ import annotations

import csv
import importlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
import warnings
import zipfile
from pathlib import Path
from typing import Any

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
OUTPUTS_DIR = PROJECT_ROOT / "outputs" / "preliminary_results"
RUNS_DIR = OUTPUTS_DIR / "runs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"
METADATA_DIR = OUTPUTS_DIR / "metadata"
EXTERNAL_DIR = PROJECT_ROOT / "external"
TGM_ROOT = EXTERNAL_DIR / "tgm"

for d in [OUTPUTS_DIR, RUNS_DIR, FIGURES_DIR, TABLES_DIR, METADATA_DIR, EXTERNAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Make vendored TGM importable even if the notebook kernel isn't
# using the same environment that `pip install -e` targeted.
if TGM_ROOT.exists():
    sys.path.insert(0, str(TGM_ROOT))
    importlib.invalidate_caches()

def run(
    cmd: list[str],
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
    check: bool = True,
    capture: bool = False,
) -> subprocess.CompletedProcess:
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd is not None else None,
        env=env,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=check,
        capture_output=capture,
    )

print("PROJECT_ROOT =", PROJECT_ROOT)
print("OUTPUTS_DIR  =", OUTPUTS_DIR)
print("PYTHON       =", sys.executable)
print("TGM_ROOT     =", TGM_ROOT)


PROJECT_ROOT = c:\Users\dansu\OneDrive\Bureau\mcgill courses\COMP511\Project
OUTPUTS_DIR  = c:\Users\dansu\OneDrive\Bureau\mcgill courses\COMP511\Project\outputs\preliminary_results
PYTHON       = c:\Users\dansu\AppData\Local\Programs\Python\Python311\python.exe
TGM_ROOT     = c:\Users\dansu\OneDrive\Bureau\mcgill courses\COMP511\Project\external\tgm


In [2]:
FAST_MODE = True
DEVICE_REQUEST = "auto"

# Reuse cached runs by default so the notebook can be rerun safely.
OVERWRITE = False
REUSE_FAILED = True

# Reproduce the same TGM code base used by the existing cached runs.
TGM_REPO_URL = "https://github.com/tgm-team/tgm.git"
TGM_COMMIT = "060030364b71c09f8ff093a82bbf740c2301fb99"

# Restore prior successful / failed runs if the user uploaded the saved bundles.
AUTO_RESTORE_RESULT_BUNDLES = True
RESULT_BUNDLES = ["base3_edgebank.zip", "tgn.zip"]

# The uploaded helper file may be named `sitecustomize (1).py`;
# copy it to the canonical importable name without changing its contents.
AUTO_COPY_SITECUSTOMIZE = True
SITECUSTOMIZE_CANDIDATES = ["sitecustomize.py", "sitecustomize (1).py"]

NUM_TIME_BINS = 50
SEEDS = [1337] if FAST_MODE else [1337, 2026, 7]

# Full visible benchmark scope kept in the notebook for transparency.
VISIBLE_DATASETS = ["tgbl-wiki", "tgbl-review", "tgbl-coin"]
VISIBLE_MODELS = ["edgebank", "base3", "tgn", "dygformer", "graphmixer"]

# Narrowed final-study scope: run only the feasible experiments that matter.
MAIN_PERFORMANCE_DATASETS = ["tgbl-wiki", "tgbl-review"]
FEASIBILITY_ONLY_DATASETS = ["tgbl-coin"]

ACTIVE_EXPERIMENTS: list[tuple[str, str]] = [
    ("tgbl-wiki", "edgebank"),
    ("tgbl-wiki", "base3"),
    ("tgbl-wiki", "tgn"),
    ("tgbl-wiki", "graphmixer"),
    ("tgbl-review", "edgebank"),
    ("tgbl-review", "base3"),
    ("tgbl-review", "graphmixer"),
]

# Metrics are computed only for the datasets used in the main predictive study
# unless you explicitly want to include feasibility-only datasets too.
INCLUDE_FEASIBILITY_ONLY_DATASET_METRICS = False
METRIC_DATASETS = list(MAIN_PERFORMANCE_DATASETS)
if INCLUDE_FEASIBILITY_ONLY_DATASET_METRICS:
    for ds in FEASIBILITY_ONLY_DATASETS:
        if ds not in METRIC_DATASETS:
            METRIC_DATASETS.append(ds)

MODEL_TO_SCRIPT = {
    "edgebank": "examples/linkproppred/edgebank.py",
    "base3": "examples/linkproppred/base3.py",
    "tgn": "examples/linkproppred/tgn.py",
    "dygformer": "examples/linkproppred/dygformer.py",
    "graphmixer": "examples/linkproppred/graphmixer.py",
}

MODEL_NEEDS_DEVICE = {
    "edgebank": False,
    "base3": False,
    "tgn": True,
    "dygformer": True,
    "graphmixer": True,
}

MODEL_CLI_ARGS = {
    "edgebank": [
        ("--bsize", 200),
        ("--window-ratio", 0.15),
        ("--memory-mode", "fixed"),
        ("--pos-prob", 1.0),
    ],
    "base3": [
        ("--bsize", 200),
        ("--window-ratio", 0.15),
        ("--k", 50),
        ("--co-occur", 0.8),
        ("--pos-prob", 1.0),
    ],
    "tgn": [
        ("--bsize", 200),
        ("--epochs", 10 if FAST_MODE else 30),
        ("--time-dim", 100),
        ("--embed-dim", 100),
        ("--memory-dim", 100),
        ("--n-nbrs", [10]),
    ],
    "dygformer": [
        ("--bsize", 200),
        ("--epochs", 3 if FAST_MODE else 10),
        ("--lr", 0.0001),
        ("--max_sequence_length", 32),
        ("--time_dim", 100),
        ("--embed_dim", 172),
        ("--node_dim", 128),
        ("--channel-embedding-dim", 50),
        ("--patch-size", 1),
        ("--num_layers", 2),
        ("--num_heads", 2),
        ("--num-channels", 4),
    ],
    "graphmixer": [
        ("--bsize", 200),
        ("--epochs", 3 if FAST_MODE else 10),
        ("--lr", 0.0002),
        ("--n-nbrs", 20),
        ("--time-dim", 100),
        ("--embed-dim", 128),
        ("--node-dim", 100),
        ("--time-gap", 2000),
        ("--token-dim-expansion", 0.5),
        ("--channel-dim-expansion", 4.0),
    ],
}

ACTIVE_EXPERIMENT_SET = {(dataset, model, seed) for dataset, model in ACTIVE_EXPERIMENTS for seed in SEEDS}
VISIBLE_EXPERIMENT_SET = {(dataset, model, seed) for dataset in VISIBLE_DATASETS for model in VISIBLE_MODELS for seed in SEEDS}
MAIN_PERFORMANCE_EXPERIMENT_SET = {
    (dataset, model, seed)
    for dataset, model in ACTIVE_EXPERIMENTS
    if dataset in MAIN_PERFORMANCE_DATASETS
    for seed in SEEDS
}

config = {
    "FAST_MODE": FAST_MODE,
    "DEVICE_REQUEST": DEVICE_REQUEST,
    "OVERWRITE": OVERWRITE,
    "REUSE_FAILED": REUSE_FAILED,
    "TGM_REPO_URL": TGM_REPO_URL,
    "TGM_COMMIT": TGM_COMMIT,
    "AUTO_RESTORE_RESULT_BUNDLES": AUTO_RESTORE_RESULT_BUNDLES,
    "RESULT_BUNDLES": RESULT_BUNDLES,
    "AUTO_COPY_SITECUSTOMIZE": AUTO_COPY_SITECUSTOMIZE,
    "SITECUSTOMIZE_CANDIDATES": SITECUSTOMIZE_CANDIDATES,
    "NUM_TIME_BINS": NUM_TIME_BINS,
    "SEEDS": SEEDS,
    "VISIBLE_DATASETS": VISIBLE_DATASETS,
    "VISIBLE_MODELS": VISIBLE_MODELS,
    "MAIN_PERFORMANCE_DATASETS": MAIN_PERFORMANCE_DATASETS,
    "FEASIBILITY_ONLY_DATASETS": FEASIBILITY_ONLY_DATASETS,
    "ACTIVE_EXPERIMENTS": ACTIVE_EXPERIMENTS,
    "METRIC_DATASETS": METRIC_DATASETS,
}
(METADATA_DIR / "run_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print(json.dumps(config, indent=2))


{
  "FAST_MODE": true,
  "DEVICE_REQUEST": "auto",
  "OVERWRITE": false,
  "REUSE_FAILED": true,
  "TGM_REPO_URL": "https://github.com/tgm-team/tgm.git",
  "TGM_COMMIT": "060030364b71c09f8ff093a82bbf740c2301fb99",
  "AUTO_RESTORE_RESULT_BUNDLES": true,
  "RESULT_BUNDLES": [
    "base3_edgebank.zip",
    "tgn.zip"
  ],
  "AUTO_COPY_SITECUSTOMIZE": true,
  "SITECUSTOMIZE_CANDIDATES": [
    "sitecustomize.py",
    "sitecustomize (1).py"
  ],
  "NUM_TIME_BINS": 50,
  "SEEDS": [
    1337
  ],
  "VISIBLE_DATASETS": [
    "tgbl-wiki",
    "tgbl-review",
    "tgbl-coin"
  ],
  "VISIBLE_MODELS": [
    "edgebank",
    "base3",
    "tgn",
    "dygformer",
    "graphmixer"
  ],
  "MAIN_PERFORMANCE_DATASETS": [
    "tgbl-wiki",
    "tgbl-review"
  ],
  "FEASIBILITY_ONLY_DATASETS": [
    "tgbl-coin"
  ],
  "ACTIVE_EXPERIMENTS": [
    [
      "tgbl-wiki",
      "edgebank"
    ],
    [
      "tgbl-wiki",
      "base3"
    ],
    [
      "tgbl-wiki",
      "tgn"
    ],
    [
      "tgbl-wiki",
      "g

In [3]:
if shutil.which("git") is None:
    raise RuntimeError("git is required but was not found on PATH.")

def ensure_sitecustomize_file() -> Path | None:
    preferred = PROJECT_ROOT / "sitecustomize.py"
    if preferred.exists():
        print("[INFO] Using existing sitecustomize.py")
        return preferred

    if not AUTO_COPY_SITECUSTOMIZE:
        print("[INFO] AUTO_COPY_SITECUSTOMIZE is disabled.")
        return None

    for candidate_name in SITECUSTOMIZE_CANDIDATES:
        candidate = PROJECT_ROOT / candidate_name
        if candidate.exists() and candidate != preferred:
            shutil.copy2(candidate, preferred)
            print(f"[INFO] Copied {candidate.name} -> {preferred.name}")
            return preferred

    print("[INFO] No sitecustomize helper file found in the project root.")
    return None

def restore_result_bundles() -> list[dict[str, str]]:
    restored: list[dict[str, str]] = []
    if not AUTO_RESTORE_RESULT_BUNDLES:
        print("[INFO] AUTO_RESTORE_RESULT_BUNDLES is disabled.")
        return restored

    for bundle_name in RESULT_BUNDLES:
        bundle_path = PROJECT_ROOT / bundle_name
        if not bundle_path.exists():
            print(f"[INFO] Bundle not found, skipping: {bundle_name}")
            continue

        with zipfile.ZipFile(bundle_path, "r") as zf:
            zf.extractall(PROJECT_ROOT)
        restored.append({"bundle": bundle_name, "status": "restored"})
        print(f"[INFO] Restored cached runs from: {bundle_name}")

    return restored

sitecustomize_path = ensure_sitecustomize_file()
restored_bundles = restore_result_bundles()
(METADATA_DIR / "restored_bundles.json").write_text(json.dumps(restored_bundles, indent=2), encoding="utf-8")

# Keep installs explicit and notebook-friendly.
# NOTE: TGM's `pyproject.toml` uses dependency groups (uv/poetry style), not pip extras.
# So `pip install -e .[examples]` will NOT install `py-tgb`, etc.
run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])

# Scientific stack (used throughout this notebook)
run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "numpy",
        "pandas",
        "scipy",
        "matplotlib",
    ]
)

# Core ML + benchmark deps used by TGM examples and by `DGData.from_tgb`
# Install torch first, then install the PyG compiled extension wheels that
# match your exact torch build.
run([sys.executable, "-m", "pip", "install", "--upgrade", "torch"])

import torch

torch_ver = torch.__version__.split("+")[0]
if torch.version.cuda is None:
    torch_build = "cpu"
else:
    cuda_digits = "".join(str(torch.version.cuda).split("."))
    torch_build = f"cu{cuda_digits}"

pyg_wheel_index = f"https://data.pyg.org/whl/torch-{torch_ver}+{torch_build}.html"
print("[INFO] torch version =", torch.__version__)
print("[INFO] torch build   =", torch_build)
print("[INFO] PyG wheels    =", pyg_wheel_index)

# Install PyG extension packages explicitly to avoid broken/mismatched binaries.
run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "pyg-lib",
        "torch-scatter",
        "torch-sparse",
        "torch-cluster",
        "torch-spline-conv",
        "-f",
        pyg_wheel_index,
    ]
)

run([sys.executable, "-m", "pip", "install", "--upgrade", "torch-geometric"])

run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "py-tgb",
        "tgb-seq",
        "torchmetrics",
        "tqdm",
    ]
)

# Clone/update TGM at the exact commit used for the cached legacy runs.
if TGM_ROOT.exists() and (TGM_ROOT / ".git").exists():
    run(["git", "fetch", "--all"], cwd=TGM_ROOT)
else:
    if TGM_ROOT.exists():
        shutil.rmtree(TGM_ROOT)
    run(["git", "clone", TGM_REPO_URL, str(TGM_ROOT)])

run(["git", "checkout", "--force", TGM_COMMIT], cwd=TGM_ROOT)

# Editable install of the local repo (best effort).
# If editable metadata doesn't get picked up by this kernel on Windows,
# we still make it importable via sys.path below.
run([sys.executable, "-m", "pip", "install", "-e", str(TGM_ROOT)])

# Ensure the vendored repo is importable in this notebook kernel.
import importlib

sys.path.insert(0, str(TGM_ROOT))
importlib.invalidate_caches()

# Sanity checks: fail early with clear errors + diagnostics
print("[ENV] python             =", sys.executable)
print("[ENV] TGM_ROOT           =", TGM_ROOT)
print("[ENV] exists             =", TGM_ROOT.exists())
print("[ENV] has_tgm            =", (TGM_ROOT / "tgm").exists())
print("[ENV] sitecustomize_path =", sitecustomize_path)

for mod in ["tgm", "tgb"]:
    try:
        m = importlib.import_module(mod)
        print(f"[OK] import {mod}")
        if mod == "tgm":
            print("[OK] tgm file =", getattr(m, "__file__", None))
    except Exception as e:
        raise RuntimeError(
            f"Failed to import `{mod}` after installation/path setup. "
            f"Kernel Python: {sys.executable}\n"
            f"TGM_ROOT: {TGM_ROOT} (exists={TGM_ROOT.exists()}, has_tgm_dir={(TGM_ROOT / 'tgm').exists()})\n"
            f"Error: {type(e).__name__}: {e}"
        )


[INFO] Using existing sitecustomize.py
[INFO] Restored cached runs from: base3_edgebank.zip
[INFO] Restored cached runs from: tgn.zip
$ c:\Users\dansu\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip setuptools wheel
$ c:\Users\dansu\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade numpy pandas scipy matplotlib
$ c:\Users\dansu\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade torch
[INFO] torch version = 2.11.0+cpu
[INFO] torch build   = cpu
[INFO] PyG wheels    = https://data.pyg.org/whl/torch-2.11.0+cpu.html
$ c:\Users\dansu\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.11.0+cpu.html
$ c:\Users\dansu\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade torch-geometric
$ c:\Users\dansu\AppData\Local\Programs\Python\Python311\python.exe -m pip insta

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from tgm import DGraph
from tgm.data import DGData

if DEVICE_REQUEST == "auto":
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
else:
    DEVICE = DEVICE_REQUEST

git_hash = run(["git", "rev-parse", "HEAD"], cwd=TGM_ROOT, capture=True).stdout.strip()
versions = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "torch": torch.__version__,
    "device": DEVICE,
    "tgm_git_hash": git_hash,
}
(METADATA_DIR / "environment.json").write_text(json.dumps(versions, indent=2), encoding="utf-8")
print(json.dumps(versions, indent=2))


$ git rev-parse HEAD
{
  "python": "3.11.3",
  "numpy": "2.4.4",
  "pandas": "3.0.2",
  "torch": "2.11.0+cpu",
  "device": "cpu",
  "tgm_git_hash": "060030364b71c09f8ff093a82bbf740c2301fb99"
}


In [5]:
VAL_RE = re.compile(r"Validation .*?=([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)")
TEST_RE = re.compile(r"Test .*?=([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)")

def flatten_cli_args(arg_pairs: list[tuple[str, Any]]) -> list[str]:
    args: list[str] = []
    for flag, value in arg_pairs:
        if isinstance(value, (list, tuple)):
            args.append(flag)
            args.extend(str(v) for v in value)
        else:
            args.extend([flag, str(value)])
    return args

def build_command(model: str, dataset: str, seed: int, log_file: Path) -> list[str]:
    script_path = TGM_ROOT / MODEL_TO_SCRIPT[model]
    if not script_path.exists():
        raise FileNotFoundError(f"Missing script for {model}: {script_path}")

    cmd = [
        sys.executable,
        str(script_path),
        "--dataset",
        dataset,
        "--seed",
        str(seed),
        "--log-file-path",
        str(log_file),
    ]

    if MODEL_NEEDS_DEVICE[model]:
        cmd.extend(["--device", DEVICE])

    cmd.extend(flatten_cli_args(MODEL_CLI_ARGS[model]))
    return cmd

def parse_metrics_from_text(text: str) -> tuple[float | None, float | None]:
    val_matches = VAL_RE.findall(text)
    test_matches = TEST_RE.findall(text)

    val_mrr = max((float(v) for v in val_matches), default=None)
    test_mrr = float(test_matches[-1]) if test_matches else None
    return val_mrr, test_mrr

def parse_metrics_from_files(log_file: Path, stdout_file: Path) -> tuple[float | None, float | None]:
    texts = []
    for path in [log_file, stdout_file]:
        if path.exists():
            texts.append(path.read_text(encoding="utf-8", errors="ignore"))
    full_text = "\n".join(texts)
    return parse_metrics_from_text(full_text)

def infer_failure_type(text: str, returncode: int, test_mrr: float | None) -> str | None:
    if returncode == 0 and test_mrr is not None:
        return None

    lowered = text.lower()
    if "out of memory" in lowered or "cuda oom" in lowered or "oom" in lowered:
        return "oom"
    if "killed" in lowered:
        return "killed"
    if "timed out" in lowered or "timeout" in lowered:
        return "timeout"
    if "no module named" in lowered or "modulenotfounderror" in lowered:
        return "missing_dependency"
    if "filenotfounderror" in lowered or "missing script" in lowered:
        return "missing_file"
    if returncode == 0 and test_mrr is None:
        return "missing_metric"
    return "failed"

def metadata_path_for(model: str, dataset: str, seed: int) -> Path:
    return RUNS_DIR / model / dataset / f"seed_{seed}" / "metadata.json"

def load_all_cached_metadata() -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for metadata_file in sorted(RUNS_DIR.rglob("metadata.json")):
        row = json.loads(metadata_file.read_text(encoding="utf-8"))
        row.setdefault("cached", True)
        row.setdefault("command", "")
        row.setdefault("log_file", str(metadata_file.parent / "tgm.log"))
        row.setdefault("stdout_file", str(metadata_file.parent / "stdout.txt"))
        row.setdefault("stderr_file", str(metadata_file.parent / "stderr.txt"))
        row.setdefault("returncode", None)
        row.setdefault("runtime_seconds", float("nan"))
        row.setdefault("val_mrr", None)
        row.setdefault("test_mrr", None)
        row.setdefault("device", DEVICE)

        dataset = row.get("dataset")
        model = row.get("model")
        seed = row.get("seed")
        row["in_visible_scope"] = (dataset, model, seed) in VISIBLE_EXPERIMENT_SET
        row["in_active_plan"] = (dataset, model, seed) in ACTIVE_EXPERIMENT_SET
        row["in_main_performance_plan"] = (dataset, model, seed) in MAIN_PERFORMANCE_EXPERIMENT_SET

        if not row.get("failure_type") and row.get("status") != "ok":
            text_parts = []
            for key in ["stdout_file", "stderr_file", "log_file"]:
                path = Path(row[key])
                if path.exists():
                    text_parts.append(path.read_text(encoding="utf-8", errors="ignore"))
            row["failure_type"] = infer_failure_type("\n".join(text_parts), int(row.get("returncode") or -1), row.get("test_mrr"))

        rows.append(row)
    return rows

def run_single_model(model: str, dataset: str, seed: int) -> dict[str, Any]:
    run_dir = RUNS_DIR / model / dataset / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    log_file = run_dir / "tgm.log"
    stdout_file = run_dir / "stdout.txt"
    stderr_file = run_dir / "stderr.txt"
    metadata_file = run_dir / "metadata.json"

    # Cache behavior:
    # - Always reuse successful runs unless OVERWRITE=True.
    # - Reuse failed runs too when REUSE_FAILED=True so restored bundles stay visible
    #   in feasibility analysis without being rerun.
    if metadata_file.exists() and not OVERWRITE:
        row = json.loads(metadata_file.read_text(encoding="utf-8"))
        if row.get("status") == "ok" or REUSE_FAILED:
            row["cached"] = True
            return row

    cmd = build_command(model=model, dataset=dataset, seed=seed, log_file=log_file)
    start = time.perf_counter()

    # Ensure project-local `sitecustomize.py` is importable in the subprocess.
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    project_root = str(PROJECT_ROOT)
    env["PYTHONPATH"] = project_root + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")

    proc = subprocess.run(
        cmd,
        cwd=str(TGM_ROOT),
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
        env=env,
    )
    elapsed = time.perf_counter() - start

    stdout_file.write_text(proc.stdout or "", encoding="utf-8")
    stderr_file.write_text(proc.stderr or "", encoding="utf-8")

    val_mrr, test_mrr = parse_metrics_from_files(log_file, stdout_file)
    failure_type = infer_failure_type((proc.stdout or "") + "\n" + (proc.stderr or ""), proc.returncode, test_mrr)

    row = {
        "model": model,
        "dataset": dataset,
        "seed": seed,
        "device": DEVICE,
        "status": "ok" if proc.returncode == 0 and test_mrr is not None else "failed",
        "failure_type": failure_type,
        "returncode": proc.returncode,
        "val_mrr": val_mrr,
        "test_mrr": test_mrr,
        "runtime_seconds": elapsed,
        "command": " ".join(cmd),
        "log_file": str(log_file),
        "stdout_file": str(stdout_file),
        "stderr_file": str(stderr_file),
        "cached": False,
        "in_visible_scope": (dataset, model, seed) in VISIBLE_EXPERIMENT_SET,
        "in_active_plan": (dataset, model, seed) in ACTIVE_EXPERIMENT_SET,
        "in_main_performance_plan": (dataset, model, seed) in MAIN_PERFORMANCE_EXPERIMENT_SET,
    }
    metadata_file.write_text(json.dumps(row, indent=2), encoding="utf-8")
    return row


In [6]:
executed_rows: list[dict[str, Any]] = []

print("[PLAN] Active experiments to execute:")
for dataset, model in ACTIVE_EXPERIMENTS:
    print(f"  - {dataset:12s} | {model}")

for dataset, model in ACTIVE_EXPERIMENTS:
    for seed in SEEDS:
        print(f"[RUN] dataset={dataset} model={model} seed={seed} device={DEVICE}")
        row = run_single_model(model=model, dataset=dataset, seed=seed)
        executed_rows.append(row)
        print(
            f"      status={row['status']} "
            f"val_mrr={row['val_mrr']} test_mrr={row['test_mrr']} "
            f"runtime={row['runtime_seconds']:.2f}s cached={row['cached']}"
        )

all_run_rows = load_all_cached_metadata()
raw_results_df = pd.DataFrame(all_run_rows).sort_values(["dataset", "model", "seed"]).reset_index(drop=True)

# Keep only the benchmark scope exposed in the notebook.
raw_results_df = raw_results_df[raw_results_df["in_visible_scope"]].copy().reset_index(drop=True)

# Predictive analysis uses only successful runs from the narrowed main plan.
performance_results_df = raw_results_df[
    (raw_results_df["status"] == "ok") & (raw_results_df["in_main_performance_plan"])
].copy()

# Feasibility analysis keeps every attempted visible pair, including restored failures.
feasibility_results_df = raw_results_df.copy()
failed_results_df = feasibility_results_df[feasibility_results_df["status"] != "ok"].copy()

raw_results_path = OUTPUTS_DIR / "raw_results.csv"
performance_results_path = OUTPUTS_DIR / "performance_scope_results.csv"
feasibility_results_path = OUTPUTS_DIR / "feasibility_results.csv"

raw_results_df.to_csv(raw_results_path, index=False)
performance_results_df.to_csv(performance_results_path, index=False)
feasibility_results_df.to_csv(feasibility_results_path, index=False)

if not failed_results_df.empty:
    failed_results_df.to_csv(OUTPUTS_DIR / "failed_runs.csv", index=False)

summary_results_df = (
    performance_results_df.groupby(["dataset", "model"], as_index=False)
    .agg(
        n_runs=("seed", "count"),
        val_mrr_mean=("val_mrr", "mean"),
        val_mrr_std=("val_mrr", "std"),
        test_mrr_mean=("test_mrr", "mean"),
        test_mrr_std=("test_mrr", "std"),
        runtime_seconds_mean=("runtime_seconds", "mean"),
        runtime_seconds_std=("runtime_seconds", "std"),
    )
    .sort_values(["dataset", "model"])
)
for col in ["val_mrr_std", "test_mrr_std", "runtime_seconds_std"]:
    if col in summary_results_df.columns:
        summary_results_df[col] = summary_results_df[col].fillna(0.0)

summary_results_path = OUTPUTS_DIR / "summary_results.csv"
summary_results_df.to_csv(summary_results_path, index=False)

coverage_df = (
    feasibility_results_df.sort_values(["dataset", "model", "seed"])
    [["dataset", "model", "seed", "status", "failure_type", "runtime_seconds", "cached", "in_active_plan"]]
    .reset_index(drop=True)
)
coverage_path = OUTPUTS_DIR / "coverage_results.csv"
coverage_df.to_csv(coverage_path, index=False)

print(f"Saved: {raw_results_path}")
print(f"Saved: {performance_results_path}")
print(f"Saved: {feasibility_results_path}")
print(f"Saved: {summary_results_path}")
print(f"Saved: {coverage_path}")
if not failed_results_df.empty:
    print(f"Saved: {OUTPUTS_DIR / 'failed_runs.csv'}")

summary_results_df


[PLAN] Active experiments to execute:
  - tgbl-wiki    | edgebank
  - tgbl-wiki    | base3
  - tgbl-wiki    | tgn
  - tgbl-wiki    | graphmixer
  - tgbl-review  | edgebank
  - tgbl-review  | base3
  - tgbl-review  | graphmixer
[RUN] dataset=tgbl-wiki model=edgebank seed=1337 device=cpu
      status=ok val_mrr=0.5997 test_mrr=0.571 runtime=64.61s cached=True
[RUN] dataset=tgbl-wiki model=base3 seed=1337 device=cpu
      status=ok val_mrr=0.5997 test_mrr=0.571 runtime=621.76s cached=True
[RUN] dataset=tgbl-wiki model=tgn seed=1337 device=cpu
      status=ok val_mrr=0.4835 test_mrr=0.4586 runtime=768.30s cached=True
[RUN] dataset=tgbl-wiki model=graphmixer seed=1337 device=cpu
      status=ok val_mrr=0.2191 test_mrr=0.1852 runtime=6008.49s cached=True
[RUN] dataset=tgbl-review model=edgebank seed=1337 device=cpu
      status=ok val_mrr=0.0242 test_mrr=0.0253 runtime=839.45s cached=True
[RUN] dataset=tgbl-review model=base3 seed=1337 device=cpu
      status=ok val_mrr=0.0242 test_mrr=0.025

,dataset,model,n_runs,val_mrr_mean,val_mrr_std,test_mrr_mean,test_mrr_std,runtime_seconds_mean,runtime_seconds_std
0,tgbl-review,base3,1,0.0242,0.0,0.0253,0.0,3606.503824,0.0
1,tgbl-review,edgebank,1,0.0242,0.0,0.0253,0.0,839.451760,0.0
2,tgbl-review,graphmixer,1,0.3434,0.0,0.4121,0.0,30199.866387,0.0
3,tgbl-wiki,base3,1,0.5997,0.0,0.5710,0.0,621.756115,0.0
4,tgbl-wiki,edgebank,1,0.5997,0.0,0.5710,0.0,64.611318,0.0
5,tgbl-wiki,graphmixer,1,0.2191,0.0,0.1852,0.0,6008.493763,0.0
6,tgbl-wiki,tgn,1,0.4835,0.0,0.4586,0.0,768.303180,0.0


In [7]:
from dataclasses import asdict, dataclass

@dataclass
class DatasetMetrics:
    dataset: str
    num_nodes: int
    num_edges: int
    num_bins: int
    surprise: float
    reoccurrence: float
    novelty: float
    edge_repeat_rate: float
    mean_active_node_ratio: float
    mean_active_edge_density: float
    degree_volatility: float
    node_lifetime_median: float
    node_lifetime_iqr: float

def to_numpy(x) -> np.ndarray:
    if hasattr(x, "detach"):
        x = x.detach()
    if hasattr(x, "cpu"):
        x = x.cpu()
    if hasattr(x, "numpy"):
        x = x.numpy()
    return np.asarray(x)

def materialize_split(data_obj) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    dg = DGraph(data_obj)
    materialized = dg.materialize(materialize_features=False)
    src = to_numpy(materialized.edge_src).astype(np.int64, copy=False)
    dst = to_numpy(materialized.edge_dst).astype(np.int64, copy=False)
    ts = to_numpy(materialized.edge_time).astype(np.float64, copy=False)
    return src, dst, ts

def edge_keys(src: np.ndarray, dst: np.ndarray, num_nodes: int) -> np.ndarray:
    base = np.uint64(num_nodes + 1)
    return src.astype(np.uint64) * base + dst.astype(np.uint64)

def split_boundaries(values: np.ndarray, num_bins: int) -> np.ndarray:
    if values.size == 0:
        return np.zeros(num_bins + 1, dtype=np.int64)
    if values.min() == values.max():
        bounds = np.zeros(num_bins + 1, dtype=np.int64)
        bounds[-1] = values.size
        return bounds
    bin_edges = np.linspace(values.min(), values.max(), num_bins + 1)
    bounds = np.searchsorted(values, bin_edges, side="left")
    bounds[0] = 0
    bounds[-1] = values.size
    return bounds.astype(np.int64)

def compute_surprise_and_reoccurrence(
    train_src: np.ndarray,
    train_dst: np.ndarray,
    test_src: np.ndarray,
    test_dst: np.ndarray,
    num_nodes: int,
) -> tuple[float, float]:
    train_u = np.unique(edge_keys(train_src, train_dst, num_nodes))
    test_u = np.unique(edge_keys(test_src, test_dst, num_nodes))
    if test_u.size == 0:
        return np.nan, np.nan
    overlap = np.intersect1d(train_u, test_u, assume_unique=True).size
    reoccurrence = overlap / test_u.size
    surprise = 1.0 - reoccurrence
    return float(surprise), float(reoccurrence)

def compute_novelty_and_repeat(edge_u64: np.ndarray, bounds: np.ndarray) -> tuple[float, float]:
    seen = np.array([], dtype=np.uint64)
    novelty_vals: list[float] = []
    repeat_vals: list[float] = []

    for i in range(len(bounds) - 1):
        start, end = int(bounds[i]), int(bounds[i + 1])
        if end <= start:
            continue

        bin_unique = np.unique(edge_u64[start:end])
        if bin_unique.size == 0:
            continue

        if seen.size == 0:
            novelty = 1.0
        else:
            seen_mask = np.isin(bin_unique, seen, assume_unique=False)
            novelty = 1.0 - float(seen_mask.mean())

        repeat_rate = 1.0 - novelty
        novelty_vals.append(novelty)
        repeat_vals.append(repeat_rate)
        seen = np.union1d(seen, bin_unique)

    if not novelty_vals:
        return np.nan, np.nan
    return float(np.mean(novelty_vals)), float(np.mean(repeat_vals))

def compute_temporal_density_and_degree(
    src: np.ndarray,
    dst: np.ndarray,
    edge_u64: np.ndarray,
    bounds: np.ndarray,
    total_nodes: int,
) -> tuple[float, float, float]:
    active_node_ratios: list[float] = []
    active_edge_densities: list[float] = []
    mean_total_degrees: list[float] = []

    for i in range(len(bounds) - 1):
        start, end = int(bounds[i]), int(bounds[i + 1])
        if end <= start:
            continue

        src_b = src[start:end]
        dst_b = dst[start:end]
        edge_u_b = np.unique(edge_u64[start:end])
        active_nodes = np.unique(np.concatenate([src_b, dst_b]))
        n_t = active_nodes.size
        if n_t == 0:
            continue

        active_node_ratios.append(n_t / total_nodes)

        possible_edges = n_t * (n_t - 1)
        density_t = 0.0 if possible_edges == 0 else edge_u_b.size / possible_edges
        active_edge_densities.append(float(density_t))

        _, degree_counts = np.unique(np.concatenate([src_b, dst_b]), return_counts=True)
        mean_total_degrees.append(float(degree_counts.mean()))

    mean_node_ratio = float(np.mean(active_node_ratios)) if active_node_ratios else np.nan
    mean_edge_density = float(np.mean(active_edge_densities)) if active_edge_densities else np.nan

    if len(mean_total_degrees) >= 2 and np.mean(mean_total_degrees) > 0:
        degree_volatility = float(np.std(mean_total_degrees, ddof=1) / np.mean(mean_total_degrees))
    else:
        degree_volatility = 0.0

    return mean_node_ratio, mean_edge_density, degree_volatility

def compute_node_lifetime_stats(src: np.ndarray, dst: np.ndarray, ts: np.ndarray) -> tuple[float, float]:
    all_nodes = np.concatenate([src, dst])
    all_times = np.concatenate([ts, ts])

    if all_nodes.size == 0:
        return np.nan, np.nan

    order = np.argsort(all_nodes, kind="mergesort")
    nodes_sorted = all_nodes[order]
    times_sorted = all_times[order]

    group_starts = np.r_[0, np.flatnonzero(nodes_sorted[1:] != nodes_sorted[:-1]) + 1]
    mins = np.minimum.reduceat(times_sorted, group_starts)
    maxs = np.maximum.reduceat(times_sorted, group_starts)
    lifetimes = maxs - mins

    median = float(np.median(lifetimes))
    q1 = float(np.quantile(lifetimes, 0.25))
    q3 = float(np.quantile(lifetimes, 0.75))
    return median, float(q3 - q1)

def compute_dataset_metrics(dataset: str, num_bins: int) -> DatasetMetrics:
    full_data = DGData.from_tgb(dataset)
    train_data, val_data, test_data = full_data.split()

    train_src, train_dst, train_ts = materialize_split(train_data)
    val_src, val_dst, val_ts = materialize_split(val_data)
    test_src, test_dst, test_ts = materialize_split(test_data)

    src = np.concatenate([train_src, val_src, test_src])
    dst = np.concatenate([train_dst, val_dst, test_dst])
    ts = np.concatenate([train_ts, val_ts, test_ts])

    order = np.argsort(ts, kind="mergesort")
    src = src[order]
    dst = dst[order]
    ts = ts[order]

    num_nodes = int(full_data.num_nodes)
    edge_u64 = edge_keys(src, dst, num_nodes)
    bounds = split_boundaries(ts, num_bins)

    surprise, reoccurrence = compute_surprise_and_reoccurrence(
        train_src, train_dst, test_src, test_dst, num_nodes
    )
    novelty, edge_repeat_rate = compute_novelty_and_repeat(edge_u64, bounds)
    mean_active_node_ratio, mean_active_edge_density, degree_volatility = compute_temporal_density_and_degree(
        src, dst, edge_u64, bounds, num_nodes
    )
    node_lifetime_median, node_lifetime_iqr = compute_node_lifetime_stats(src, dst, ts)

    return DatasetMetrics(
        dataset=dataset,
        num_nodes=num_nodes,
        num_edges=int(src.size),
        num_bins=num_bins,
        surprise=surprise,
        reoccurrence=reoccurrence,
        novelty=novelty,
        edge_repeat_rate=edge_repeat_rate,
        mean_active_node_ratio=mean_active_node_ratio,
        mean_active_edge_density=mean_active_edge_density,
        degree_volatility=degree_volatility,
        node_lifetime_median=node_lifetime_median,
        node_lifetime_iqr=node_lifetime_iqr,
    )


In [8]:
dataset_metrics_rows = []

for dataset in METRIC_DATASETS:
    print(f"[METRICS] dataset={dataset}")
    metrics = compute_dataset_metrics(dataset=dataset, num_bins=NUM_TIME_BINS)
    dataset_metrics_rows.append(asdict(metrics))

dataset_metrics_df = pd.DataFrame(dataset_metrics_rows).sort_values("dataset")
dataset_metrics_path = OUTPUTS_DIR / "dataset_metrics.csv"
dataset_metrics_df.to_csv(dataset_metrics_path, index=False)

print(f"Saved: {dataset_metrics_path}")
dataset_metrics_df


[METRICS] dataset=tgbl-wiki
[METRICS] dataset=tgbl-review
Saved: c:\Users\dansu\OneDrive\Bureau\mcgill courses\COMP511\Project\outputs\preliminary_results\dataset_metrics.csv


,dataset,num_nodes,num_edges,num_bins,surprise,reoccurrence,novelty,edge_repeat_rate,mean_active_node_ratio,mean_active_edge_density,degree_volatility,node_lifetime_median,node_lifetime_iqr
1,tgbl-review,352637,4873540,50,0.999533,0.000467,0.998841,0.001159,0.152075,0.000501,0.416564,158976000.0,142905600.0
0,tgbl-wiki,9227,157474,50,0.617429,0.382571,0.431438,0.568562,0.116747,0.000758,0.095021,76457.0,1543494.5


In [9]:
merged_results_df = summary_results_df.merge(dataset_metrics_df, on="dataset", how="left")
merged_results_path = OUTPUTS_DIR / "merged_results_with_metrics.csv"
merged_results_df.to_csv(merged_results_path, index=False)

def fmt_mean_std(mean: float, std: float) -> str:
    if pd.isna(mean):
        return "--"
    if pd.isna(std) or std == 0:
        return f"{mean:.4f}"
    return f"{mean:.4f} \\pm {std:.4f}"

dataset_metrics_table_df = dataset_metrics_df[
    [
        "dataset",
        "surprise",
        "reoccurrence",
        "novelty",
        "edge_repeat_rate",
        "mean_active_node_ratio",
        "mean_active_edge_density",
        "degree_volatility",
        "node_lifetime_median",
    ]
].copy().sort_values("dataset")

dataset_metrics_table_df.to_csv(TABLES_DIR / "dataset_metrics_table.csv", index=False)
dataset_metrics_table_df.to_latex(
    TABLES_DIR / "dataset_metrics_table.tex",
    index=False,
    float_format=lambda x: f"{x:.4f}",
    caption="Temporal graph metrics used in the narrowed predictive sensitivity analysis.",
    label="tab:dataset_metrics",
)

perf_table_df = summary_results_df.copy()
perf_table_df["test_mrr_display"] = perf_table_df.apply(
    lambda r: fmt_mean_std(r["test_mrr_mean"], r["test_mrr_std"]),
    axis=1,
)
perf_table_wide = perf_table_df.pivot(
    index="dataset", columns="model", values="test_mrr_display"
).sort_index().reset_index()

perf_table_wide.to_csv(TABLES_DIR / "model_performance_table.csv", index=False)
perf_table_wide.to_latex(
    TABLES_DIR / "model_performance_table.tex",
    index=False,
    escape=False,
    caption="Test MRR across the feasible model-dataset pairs kept in the narrowed final study.",
    label="tab:model_perf",
)

feasibility_table_df = feasibility_results_df[
    ["dataset", "model", "status", "failure_type", "test_mrr", "runtime_seconds", "cached", "in_active_plan"]
].copy().sort_values(["dataset", "model"])

feasibility_table_df.to_csv(TABLES_DIR / "feasibility_table.csv", index=False)
feasibility_table_df.to_latex(
    TABLES_DIR / "feasibility_table.tex",
    index=False,
    escape=False,
    float_format=lambda x: f"{x:.4f}" if pd.notna(x) else "--",
    caption="Execution outcomes for all visible benchmark pairs, including restored failed runs used in the feasibility analysis.",
    label="tab:feasibility",
)

coverage_matrix_df = feasibility_table_df.pivot(index="dataset", columns="model", values="status").reset_index()
coverage_matrix_df.to_csv(TABLES_DIR / "coverage_matrix.csv", index=False)

print(f"Saved: {merged_results_path}")
print(f"Saved: {TABLES_DIR / 'dataset_metrics_table.csv'}")
print(f"Saved: {TABLES_DIR / 'dataset_metrics_table.tex'}")
print(f"Saved: {TABLES_DIR / 'model_performance_table.csv'}")
print(f"Saved: {TABLES_DIR / 'model_performance_table.tex'}")
print(f"Saved: {TABLES_DIR / 'feasibility_table.csv'}")
print(f"Saved: {TABLES_DIR / 'feasibility_table.tex'}")
print(f"Saved: {TABLES_DIR / 'coverage_matrix.csv'}")
merged_results_df


Saved: c:\Users\dansu\OneDrive\Bureau\mcgill courses\COMP511\Project\outputs\preliminary_results\merged_results_with_metrics.csv
Saved: c:\Users\dansu\OneDrive\Bureau\mcgill courses\COMP511\Project\outputs\preliminary_results\tables\dataset_metrics_table.csv
Saved: c:\Users\dansu\OneDrive\Bureau\mcgill courses\COMP511\Project\outputs\preliminary_results\tables\dataset_metrics_table.tex
Saved: c:\Users\dansu\OneDrive\Bureau\mcgill courses\COMP511\Project\outputs\preliminary_results\tables\model_performance_table.csv
Saved: c:\Users\dansu\OneDrive\Bureau\mcgill courses\COMP511\Project\outputs\preliminary_results\tables\model_performance_table.tex
Saved: c:\Users\dansu\OneDrive\Bureau\mcgill courses\COMP511\Project\outputs\preliminary_results\tables\feasibility_table.csv
Saved: c:\Users\dansu\OneDrive\Bureau\mcgill courses\COMP511\Project\outputs\preliminary_results\tables\feasibility_table.tex
Saved: c:\Users\dansu\OneDrive\Bureau\mcgill courses\COMP511\Project\outputs\preliminary_result

,dataset,model,n_runs,val_mrr_mean,val_mrr_std,test_mrr_mean,test_mrr_std,runtime_seconds_mean,runtime_seconds_std,num_nodes,...,num_bins,surprise,reoccurrence,novelty,edge_repeat_rate,mean_active_node_ratio,mean_active_edge_density,degree_volatility,node_lifetime_median,node_lifetime_iqr
0,tgbl-review,base3,1,0.0242,0.0,0.0253,0.0,3606.503824,0.0,352637,...,50,0.999533,0.000467,0.998841,0.001159,0.152075,0.000501,0.416564,158976000.0,142905600.0
1,tgbl-review,edgebank,1,0.0242,0.0,0.0253,0.0,839.451760,0.0,352637,...,50,0.999533,0.000467,0.998841,0.001159,0.152075,0.000501,0.416564,158976000.0,142905600.0
2,tgbl-review,graphmixer,1,0.3434,0.0,0.4121,0.0,30199.866387,0.0,352637,...,50,0.999533,0.000467,0.998841,0.001159,0.152075,0.000501,0.416564,158976000.0,142905600.0
3,tgbl-wiki,base3,1,0.5997,0.0,0.5710,0.0,621.756115,0.0,9227,...,50,0.617429,0.382571,0.431438,0.568562,0.116747,0.000758,0.095021,76457.0,1543494.5
4,tgbl-wiki,edgebank,1,0.5997,0.0,0.5710,0.0,64.611318,0.0,9227,...,50,0.617429,0.382571,0.431438,0.568562,0.116747,0.000758,0.095021,76457.0,1543494.5
5,tgbl-wiki,graphmixer,1,0.2191,0.0,0.1852,0.0,6008.493763,0.0,9227,...,50,0.617429,0.382571,0.431438,0.568562,0.116747,0.000758,0.095021,76457.0,1543494.5
6,tgbl-wiki,tgn,1,0.4835,0.0,0.4586,0.0,768.303180,0.0,9227,...,50,0.617429,0.382571,0.431438,0.568562,0.116747,0.000758,0.095021,76457.0,1543494.5


In [10]:
from scipy.stats import spearmanr

def safe_spearman(x: pd.Series, y: pd.Series) -> float:
    mask = x.notna() & y.notna()
    if mask.sum() < 3:
        return np.nan
    return float(spearmanr(x[mask], y[mask]).statistic)

metric_cols = [
    "surprise",
    "reoccurrence",
    "novelty",
    "edge_repeat_rate",
    "mean_active_node_ratio",
    "mean_active_edge_density",
    "degree_volatility",
    "node_lifetime_median",
    "node_lifetime_iqr",
]

corr_rows = []
for model, g in merged_results_df.groupby("model"):
    for metric in metric_cols:
        corr_rows.append(
            {
                "model": model,
                "metric": metric,
                "spearman_with_test_mrr": safe_spearman(g[metric], g["test_mrr_mean"]),
            }
        )

correlations_df = pd.DataFrame(corr_rows).sort_values(["model", "metric"])
correlations_path = OUTPUTS_DIR / "correlations.csv"
correlations_df.to_csv(correlations_path, index=False)

correlation_pivot = correlations_df.pivot(index="metric", columns="model", values="spearman_with_test_mrr")
print(f"Saved: {correlations_path}")
correlation_pivot


Saved: c:\Users\dansu\OneDrive\Bureau\mcgill courses\COMP511\Project\outputs\preliminary_results\correlations.csv


model,base3,edgebank,graphmixer,tgn
metric,,,,
degree_volatility,NaN,NaN,NaN,NaN
edge_repeat_rate,NaN,NaN,NaN,NaN
mean_active_edge_density,NaN,NaN,NaN,NaN
mean_active_node_ratio,NaN,NaN,NaN,NaN
node_lifetime_iqr,NaN,NaN,NaN,NaN
node_lifetime_median,NaN,NaN,NaN,NaN
novelty,NaN,NaN,NaN,NaN
reoccurrence,NaN,NaN,NaN,NaN
surprise,NaN,NaN,NaN,NaN


In [11]:
def scatter_metric_vs_mrr(
    merged: pd.DataFrame,
    metric: str,
    out_path: Path,
    title: str,
    xlabel: str,
) -> None:
    if merged.empty or metric not in merged.columns:
        return

    plt.figure(figsize=(7, 5))
    for model, g in merged.groupby("model"):
        plt.scatter(g[metric], g["test_mrr_mean"], label=model)
        for _, row in g.iterrows():
            plt.annotate(
                row["dataset"],
                (row[metric], row["test_mrr_mean"]),
                fontsize=8,
                alpha=0.85,
            )
    plt.xlabel(xlabel)
    plt.ylabel("Mean test MRR")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()

def advantage_plot(
    merged: pd.DataFrame,
    out_path: Path,
    left_model: str,
    right_model: str,
) -> None:
    if merged.empty:
        return

    pivot = merged.pivot(index="dataset", columns="model", values="test_mrr_mean")
    required = {left_model, right_model}
    if not required.issubset(set(pivot.columns)):
        return

    advantage = (pivot[left_model] - pivot[right_model]).rename("advantage")
    plot_df = merged[["dataset", "surprise"]].drop_duplicates().set_index("dataset").join(advantage).dropna()
    if plot_df.empty:
        return

    plt.figure(figsize=(7, 5))
    plt.scatter(plot_df["surprise"], plot_df["advantage"])
    for ds, row in plot_df.iterrows():
        plt.annotate(ds, (row["surprise"], row["advantage"]), fontsize=8, alpha=0.85)
    plt.axhline(0.0, linestyle="--")
    plt.xlabel("Surprise")
    plt.ylabel(f"{left_model} mean test MRR - {right_model} mean test MRR")
    plt.title(f"Model advantage vs surprise: {left_model} minus {right_model}")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()

def correlation_heatmap(corr_pivot: pd.DataFrame, out_path: Path) -> None:
    if corr_pivot.empty:
        return
    values = corr_pivot.values
    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(values, aspect="auto", vmin=-1, vmax=1)
    ax.set_xticks(range(corr_pivot.shape[1]))
    ax.set_xticklabels(list(corr_pivot.columns), rotation=45, ha="right")
    ax.set_yticks(range(corr_pivot.shape[0]))
    ax.set_yticklabels(list(corr_pivot.index))
    ax.set_title("Exploratory Spearman correlation between graph metrics and mean test MRR")
    cbar = fig.colorbar(im, ax=ax)
    cbar.ax.set_ylabel("Spearman rho", rotation=90)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()

scatter_metric_vs_mrr(
    merged_results_df,
    metric="surprise",
    out_path=FIGURES_DIR / "surprise_vs_mrr.png",
    title="Predictive sensitivity to dataset surprise",
    xlabel="Surprise",
)
scatter_metric_vs_mrr(
    merged_results_df,
    metric="novelty",
    out_path=FIGURES_DIR / "novelty_vs_mrr.png",
    title="Predictive sensitivity to dataset novelty",
    xlabel="Novelty",
)
scatter_metric_vs_mrr(
    merged_results_df,
    metric="degree_volatility",
    out_path=FIGURES_DIR / "degree_volatility_vs_mrr.png",
    title="Predictive sensitivity to degree volatility",
    xlabel="Degree volatility",
)
advantage_plot(
    merged_results_df,
    out_path=FIGURES_DIR / "graphmixer_minus_base3_vs_surprise.png",
    left_model="graphmixer",
    right_model="base3",
)
correlation_heatmap(
    correlation_pivot,
    out_path=FIGURES_DIR / "correlation_heatmap.png",
)

print("Saved figures to:", FIGURES_DIR)
sorted(p.name for p in FIGURES_DIR.iterdir())


Saved figures to: c:\Users\dansu\OneDrive\Bureau\mcgill courses\COMP511\Project\outputs\preliminary_results\figures


['correlation_heatmap.png',
 'degree_volatility_vs_mrr.png',
 'graphmixer_minus_base3_vs_surprise.png',
 'novelty_vs_mrr.png',
 'surprise_vs_mrr.png']

In [12]:
bundle_path = OUTPUTS_DIR / "preliminary_results_bundle.zip"
if bundle_path.exists():
    bundle_path.unlink()
shutil.make_archive(str(bundle_path.with_suffix("")), "zip", root_dir=OUTPUTS_DIR)
print("Saved bundle:", bundle_path)

print("\nOutput tree:")
for path in sorted(OUTPUTS_DIR.rglob("*")):
    rel = path.relative_to(OUTPUTS_DIR)
    print(rel)


Saved bundle: c:\Users\dansu\OneDrive\Bureau\mcgill courses\COMP511\Project\outputs\preliminary_results\preliminary_results_bundle.zip

Output tree:
correlations.csv
coverage_results.csv
dataset_metrics.csv
failed_runs.csv
feasibility_results.csv
figures
figures\correlation_heatmap.png
figures\degree_volatility_vs_mrr.png
figures\graphmixer_minus_base3_vs_surprise.png
figures\novelty_vs_mrr.png
figures\surprise_vs_mrr.png
merged_results_with_metrics.csv
metadata
metadata\environment.json
metadata\restored_bundles.json
metadata\run_config.json
performance_scope_results.csv
preliminary_results_bundle.zip
raw_results.csv
runs
runs\base3
runs\base3\tgbl-coin
runs\base3\tgbl-coin\seed_1337
runs\base3\tgbl-coin\seed_1337\metadata.json
runs\base3\tgbl-coin\seed_1337\stderr.txt
runs\base3\tgbl-coin\seed_1337\stdout.txt
runs\base3\tgbl-coin\seed_1337\tgm.log
runs\base3\tgbl-comment
runs\base3\tgbl-comment\seed_1337
runs\base3\tgbl-comment\seed_1337\metadata.json
runs\base3\tgbl-comment\seed_133